In [ ]:
# Lightweight path setup for local Jupyter, VS Code, or Colab runs.
# It keeps the notebook runnable whether you start it from this folder or the repo root.
from pathlib import Path
import os

FOLDER_NAME = '01-Regression-Analysis'
NOTEBOOK_NAME = 'Project07_CrossValidation.ipynb'

candidates = [
    Path.cwd(),
    Path.cwd() / FOLDER_NAME,
    Path.cwd() / "10-deep_learning" / FOLDER_NAME,
    Path.cwd().parent / FOLDER_NAME,
    Path.cwd().parent / "10-deep_learning" / FOLDER_NAME,
]

for candidate in candidates:
    if (candidate / NOTEBOOK_NAME).exists():
        os.chdir(candidate)
        break

print("Working directory:", Path.cwd().resolve())


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import sklearn.metrics as metrics

In [6]:
data = pd.read_csv("Regression_Data.csv")

In [7]:
data

,Ind_Data,Dependent_Data
0,1.1,39343
1,1.3,46205
2,1.5,37731
3,2.0,43525
4,2.2,39891
5,2.9,56642
6,3.0,60150
7,3.2,54445
8,3.2,64445
9,3.7,57189


In [8]:
data.shape

(30, 2)

In [9]:
X = data.iloc[:,:-1].values
y = data.iloc[:,-1].values
X.shape

(30, 1)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Applying Linear Regression

In [11]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

LinReg = lr.fit(X_train,y_train)

y_pred = LinReg.predict(X_test)

In [12]:
from sklearn import metrics
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

RMSE: 7059.04362190151


# Applying Cross Validation

In [14]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict

In [15]:
k = 5
cv = KFold(n_splits=k, shuffle = True)

# Build multiple linear regression model
model = LinearRegression()

#use k-fold CV to evaluate model
scores = cross_val_score(model, X, y, scoring='neg_root_mean_squared_error',
                         cv=cv, n_jobs=-1)  # n_jobs = -1 means using all processors
scores



array([-7729.72449254, -2704.03173235, -5386.42949078, -5064.4258979 ,
       -6828.64966004])

In [16]:
np.mean(np.abs(scores))

5542.652254722341

# Generating Unseen test data ( Further Compare CV with Regular train-test split )

In [17]:
test_data = np.linspace(1,11,30).reshape(-1,1)
test_data

array([[ 1.        ],
       [ 1.34482759],
       [ 1.68965517],
       [ 2.03448276],
       [ 2.37931034],
       [ 2.72413793],
       [ 3.06896552],
       [ 3.4137931 ],
       [ 3.75862069],
       [ 4.10344828],
       [ 4.44827586],
       [ 4.79310345],
       [ 5.13793103],
       [ 5.48275862],
       [ 5.82758621],
       [ 6.17241379],
       [ 6.51724138],
       [ 6.86206897],
       [ 7.20689655],
       [ 7.55172414],
       [ 7.89655172],
       [ 8.24137931],
       [ 8.5862069 ],
       [ 8.93103448],
       [ 9.27586207],
       [ 9.62068966],
       [ 9.96551724],
       [10.31034483],
       [10.65517241],
       [11.        ]])

In [18]:
test_labels = y

# Gets predictions without cross validation

In [19]:
y_pred_test = LinReg.predict(test_data)

In [20]:
print('RMSE:', np.sqrt(metrics.mean_squared_error(test_labels, y_pred_test)))
print('MAE:', metrics.mean_absolute_error(test_labels, y_pred_test))
print('R2:', metrics.r2_score(test_labels, y_pred_test))

RMSE: 9753.449010210106
MAE: 8020.953342698593
R2: 0.86905731002699


# Gets predictions with cross validation

In [21]:
y_pred_test_cv = cross_val_predict(model, test_data, test_labels,
                         cv = cv, n_jobs = -1)  # n_jobs = -1 means using all processors

In [22]:
print('RMSE:', np.sqrt(metrics.mean_squared_error(test_labels, y_pred_test_cv)))
print('MAE:', metrics.mean_absolute_error(test_labels, y_pred_test_cv))
print('R2:', metrics.r2_score(test_labels, y_pred_test_cv))

RMSE: 7737.671418859165
MAE: 6343.879364587876
R2: 0.9175889610081858
